# Blackjack: on-policy vs off-policy Monte-Carlo evaluation

A port of `blackjackOnOffPolicy.py` to a notebook.

Blackjack is a convenient test bed for off-policy learning because the model
is small enough to solve **exactly** by dynamic programming, so every
simulation-based estimate can be checked against the truth.

The plan:

1. Solve the game exactly by value iteration over
   $(\text{player sum},\ \text{dealer showing},\ \text{usable ace})$, giving
   the optimal policy $\pi^\star$ and its exact Q-factors.
2. Estimate one particular Q-factor by Monte-Carlo in three ways:
   **on-policy** (behaviour $=$ target $=\pi^\star$), **off-policy with
   ordinary importance sampling**, and **off-policy with weighted importance
   sampling**, the behaviour policy being an $\epsilon$-soft version of
   $\pi^\star$ with $\epsilon=0.6$.
3. Compare the three traces, and their rolling variance, against the exact
   value.

The Q-factor tracked is $Q\big((20,10,\text{False}),\ \text{stick}\big)$:
the player holds 20 with no usable ace, the dealer shows a 10, and the
player sticks.

::: note
As in the frozen-lake notebook, Gymnasium uses **rewards to maximise**
($+1$ win, $0$ draw, $-1$ loss) rather than the course's costs to minimise.
:::

In [ ]:
import collections, math, random
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt

try:
    import gymnasium as gym
    HAVE_GYM = True
except ImportError:
    HAVE_GYM = False
print("gymnasium available:", HAVE_GYM)

CARD_VALUES = [1,2,3,4,5,6,7,8,9,10]
CARD_PROBS  = np.array([1/13]*9 + [4/13])    # infinite deck; 10,J,Q,K all count 10
ACTIONS     = [0, 1]                         # 0 stick, 1 hit
GAMMA       = 1.0

In [ ]:
# ---- the exact model, for value iteration -------------------------------
@lru_cache(maxsize=None)
def dealer_final_probs(dealer_sum, usable_ace_flag):
    """Distribution over the dealer's final total (17..21, or 'bust')."""
    if dealer_sum >= 17:
        return {'bust': 1.0} if dealer_sum > 21 else {dealer_sum: 1.0}
    probs = collections.defaultdict(float)
    for card, p in zip(CARD_VALUES, CARD_PROBS):
        s, usable = dealer_sum, usable_ace_flag
        if card == 1:
            s, usable = (s+11, True) if s+11 <= 21 else (s+1, usable)
        else:
            s = s + card
        if s > 21 and usable:
            s, usable = s-10, False
        if s > 21:
            probs['bust'] += p
        else:
            for k, v in dealer_final_probs(s, usable).items():
                probs[k] += p*v
    return dict(probs)

ALL_STATES = tuple((ps, ds, ua) for ps in range(12, 22)
                               for ds in range(1, 11)
                               for ua in (False, True))

def get_transitions(state, action):
    player_sum, dealer_show, usable = state
    out = []
    if action == 1:                                   # hit
        for card, p in zip(CARD_VALUES, CARD_PROBS):
            s, ua = player_sum, usable
            if card == 1:
                s, ua = (s+11, True) if s+11 <= 21 else (s+1, ua)
            else:
                s = s + card
            if s > 21 and ua:
                s, ua = s-10, False
            out.append((p, None, -1.0, True) if s > 21
                       else (p, (s, dealer_show, ua), 0.0, False))
    else:                                             # stick: dealer plays
        agg = collections.defaultdict(float)
        for card, p_card in zip(CARD_VALUES, CARD_PROBS):
            d, ud = dealer_show, False
            if card == 1:
                d, ud = (d+11, True) if d+11 <= 21 else (d+1, ud)
            else:
                d = d + card
            if d > 21 and ud:
                d, ud = d-10, False
            for k, v in dealer_final_probs(d, ud).items():
                agg[k] += p_card*v
        for outcome, p_out in agg.items():
            if outcome == 'bust':          reward = 1.0
            elif outcome > player_sum:     reward = -1.0
            elif outcome < player_sum:     reward = 1.0
            else:                          reward = 0.0
            out.append((p_out, None, reward, True))
    return out

In [ ]:
def value_iteration(states, theta=1e-9):
    V = {s: 0.0 for s in states}
    while True:
        delta = 0.0
        for s in states:
            v = V[s]
            V[s] = max(sum(p*(r + (0.0 if done else GAMMA*V[ns]))
                           for p, ns, r, done in get_transitions(s, a))
                       for a in ACTIONS)
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    Q, pi = {}, {}
    for s in states:
        q = {a: sum(p*(r + (0.0 if done else GAMMA*V[ns]))
                    for p, ns, r, done in get_transitions(s, a)) for a in ACTIONS}
        for a in ACTIONS:
            Q[(s, a)] = q[a]
        pi[s] = max(ACTIONS, key=lambda a: (q[a], a))
    return V, Q, pi

V, Q_exact, pi_star = value_iteration(ALL_STATES)

record_state = (20, 10, False)
record_key   = (record_state, pi_star[record_state])
EXACT        = Q_exact[record_key]
print(f"optimal action at {record_state}: "
      f"{'hit' if pi_star[record_state] else 'stick'}")
print(f"exact Q{record_key} = {EXACT:.4f}")
print(f"exact Q({record_state}, hit)   = {Q_exact[(record_state,1)]:.4f}")

In [ ]:
# ---- episode sampling ---------------------------------------------------
class MiniBlackjack:
    """Stand-in for gymnasium's Blackjack-v1, so the notebook runs with
    nothing installed. Same infinite-deck rules and the same observation."""
    def _draw(self):   return int(np.random.choice(CARD_VALUES, p=CARD_PROBS))
    def _tot(self, s, a): return s+10 if a and s+10 <= 21 else s
    def reset(self, **kw):
        self.ps, self.pa = 0, False
        for _ in range(2):
            c = self._draw(); self.ps += c; self.pa = self.pa or c == 1
        self.ds, self.da = 0, False
        self.show = self._draw(); self.ds, self.da = self.show, self.show == 1
        self.hidden = self._draw()
        return self._obs(), {}
    def _obs(self): return (self._tot(self.ps, self.pa), self.show, self.pa)
    def step(self, a):
        if a == 1:
            c = self._draw(); self.ps += c; self.pa = self.pa or c == 1
            if self._tot(self.ps, self.pa) > 21:
                return self._obs(), -1.0, True, False, {}
            return self._obs(), 0.0, False, False, {}
        d, da = self.ds + self.hidden, self.da or self.hidden == 1
        while self._tot(d, da) < 17:
            c = self._draw(); d += c; da = da or c == 1
        pt, dt = self._tot(self.ps, self.pa), self._tot(d, da)
        r = 1.0 if (dt > 21 or pt > dt) else (0.0 if pt == dt else -1.0)
        return self._obs(), r, True, False, {}
    def close(self): pass

def make_bj():
    return gym.make("Blackjack-v1") if HAVE_GYM else MiniBlackjack()

def sample_episode(env, pi, behavior_policy=None):
    obs, _ = env.reset(); episode = []; done = False
    while not done:
        state = (obs[0], obs[1], bool(obs[2]))
        a = behavior_policy(state) if behavior_policy else pi.get(state, 1)
        obs, reward, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        episode.append((state, a, reward))
    return episode

def eps_soft(pi, eps):
    def b(s):
        return random.choice(ACTIONS) if random.random() < eps else pi.get(s, 1)
    return b

def ratio(pi, s, a, eps):
    """pi(a|s) / b(a|s) for a deterministic target and an eps-soft behaviour."""
    pi_prob = 1.0 if a == pi.get(s, 1) else 0.0
    b_prob  = eps/len(ACTIONS) + (1-eps)*(1.0 if a == pi.get(s, 1) else 0.0)
    return 0.0 if b_prob == 0 else pi_prob/b_prob

In [ ]:
def mc_on_policy(pi, n, record_sa):
    env = make_bj(); tot, cnt, trace = 0.0, 0, np.full(n, np.nan)
    for ep in range(n):
        epd = sample_episode(env, pi)
        G, seen = 0.0, set()
        for t in reversed(range(len(epd))):
            s, a, r = epd[t]; G = r + GAMMA*G
            if (s, a) not in seen:
                seen.add((s, a))
                if (s, a) == record_sa:
                    tot += G; cnt += 1
        trace[ep] = tot/cnt if cnt else np.nan
    env.close(); return trace, cnt


def mc_off_policy(pi, n, record_sa, eps, weighted, include_own_ratio=False):
    """Off-policy first-visit MC for one state-action pair.

    `weighted=False` is ordinary importance sampling, `True` is weighted.
    `include_own_ratio` reproduces the variant in the original script, which
    starts the weight product at the visited step t rather than at t+1 --
    see the discussion below.
    """
    env = make_bj(); b = eps_soft(pi, eps)
    num, den, cnt, trace = 0.0, 0.0, 0, np.full(n, np.nan)
    for ep in range(n):
        epd = sample_episode(env, pi, behavior_policy=b)
        T = len(epd)
        G = np.zeros(T); g = 0.0
        for t in range(T-1, -1, -1):
            g = epd[t][2] + GAMMA*g; G[t] = g
        ratios = [ratio(pi, s, a, eps) for (s, a, _) in epd]
        seen = set()
        for t in range(T):
            s, a, _ = epd[t]
            if (s, a) in seen:
                continue
            seen.add((s, a))
            if (s, a) == record_sa:
                W = 1.0
                for k in range(t if include_own_ratio else t+1, T):
                    W *= ratios[k]
                    if W == 0.0:
                        break
                num += W*G[t]; cnt += 1
                den += W if weighted else 1.0
        trace[ep] = num/den if den > 0 else np.nan
    env.close(); return trace, cnt

In [ ]:
N, EPS = 20000, 0.6
random.seed(0); np.random.seed(0)

tr_on,  n_on  = mc_on_policy(pi_star, N, record_key)
random.seed(1); np.random.seed(1)
tr_ord, n_ord = mc_off_policy(pi_star, N, record_key, EPS, weighted=False)
random.seed(2); np.random.seed(2)
tr_wtd, n_wtd = mc_off_policy(pi_star, N, record_key, EPS, weighted=True)
random.seed(3); np.random.seed(3)
tr_orig, n_or = mc_off_policy(pi_star, N, record_key, EPS, weighted=False,
                              include_own_ratio=True)

for name, tr, c in (("on-policy", tr_on, n_on), ("ordinary IS", tr_ord, n_ord),
                    ("weighted IS", tr_wtd, n_wtd),
                    ("ordinary IS, weight from t", tr_orig, n_or)):
    print(f"{name:28s} final {tr[-1]:+.4f}   ({c} visits)")
print(f"{'exact':28s}       {EXACT:+.4f}")

In [ ]:
def rolling_var(a, window=500):
    out = np.empty(len(a))
    for i in range(len(a)):
        w = a[max(0, i-window+1):i+1]
        out[i] = np.nanvar(w) if np.any(~np.isnan(w)) else np.nan
    return out

window = 500
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax1.axhline(EXACT, color="k", ls=":", lw=1.2, label=f"exact ({EXACT:.3f})")
ax1.plot(tr_on,  color="C0", alpha=0.9, label="On-policy")
ax1.plot(tr_wtd, color="C1", alpha=0.8, label="Off-policy (weighted IS)")
ax1.plot(tr_ord, color="C2", alpha=0.6, label="Off-policy (ordinary IS)")
ax1.set_ylabel(f"Q{record_key}"); ax1.grid(True); ax1.legend(loc="upper right")

ax2.plot(rolling_var(tr_on,  window), color="C0", label="On-policy (rolling var)")
ax2.plot(rolling_var(tr_wtd, window), color="C1", label="Off-policy weighted (rolling var)")
ax2.plot(rolling_var(tr_ord, window), color="C2", label="Off-policy ordinary (rolling var)")
ax2.set_xlabel("Episode")
ax2.set_ylabel(f"Rolling variance (window={window})")
ax2.grid(True); ax2.legend(loc="upper right")
plt.tight_layout(); plt.show()

All three estimators sit on the exact value. That is the result to take
away: off-policy evaluation recovers the value of a policy that was
**never followed**, from data generated by a different one, and the
importance ratio that makes it possible depends only on the two policies --
the model cancels.

The difference between the two off-policy estimators is variance, not
location. Ordinary importance sampling is unbiased but its weights are
unbounded, so a rare trajectory can move the average a long way. Weighted
importance sampling divides by $\sum_r\rho_r$ instead of by the count,
which is biased for a finite sample but bounded by the largest return --
and the rolling-variance panel shows the price being paid.

## A note on the ordinary-IS curve in the lecture slides

The slide for this example shows the ordinary-IS curve settling near
$0.60$ while the other two settle near $0.45$. That gap is not a property
of ordinary importance sampling -- it is unbiased, so it should land on the
same value, as it does above.

The cause is one index in `mc_off_policy_eval_trace_ordinary`:

```python
for k in range(t, T):      # weight product starts at the visited step t
    W_t *= ratios[k]
```

For a **Q**-factor the conditioning is on having taken action $a$ in state
$s$, so the product must start at $t+1$; starting at $t$ multiplies in an
extra factor $\pi(a\mid s)/b(a\mid s)$. With $\epsilon=0.6$ and two actions,
$b(\pi(s)\mid s)=0.3+0.4=0.7$, so every surviving visit is inflated by
$1/0.7\approx1.43$ -- which is exactly the direction and roughly the size of
the gap on the slide.

The weighted version in the same file does *not* have this problem, because
it updates `W` **after** the Q-update, so its product effectively starts at
$t+1$. That asymmetry between the two functions is the tell.

The cell below runs both variants side by side.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(EXACT, color="k", ls=":", lw=1.2, label=f"exact ({EXACT:.3f})")
ax.plot(tr_ord,  color="C2", label="ordinary IS, product from $t+1$ (correct)")
ax.plot(tr_orig, color="C3", label="ordinary IS, product from $t$ (as in the script)")
ax.set_xlabel("Episode"); ax.set_ylabel(f"Q{record_key}")
ax.grid(True); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

print(f"product from t+1 : {tr_ord[-1]:+.4f}")
print(f"product from t   : {tr_orig[-1]:+.4f}")
print(f"exact            : {EXACT:+.4f}")